<a href="https://www.kaggle.com/code/shamanthsr/movie-recommendation-system?scriptVersionId=319005577" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Content Based Filtering

In this notebook, we are going to learn about how the recommendation system recommend movies based on the previous movie watched or viewed.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
credits = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')
movies = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')

In [ ]:
# First three rows of credits data
credits.head(3)

In [ ]:
# shape of credits data
credits.shape

In [ ]:
# shape of movies data
movies.shape

In [ ]:
# First three rows of movies data
movies.head(3)

In [ ]:
# Renaming the column - movie_id to id in cresits dataet
credits = credits.rename(columns = {'movie_id' : 'id'})
credits.head(3)

In [ ]:
#Merging credits and movies dataset
mergedData = movies.merge(credits, on='id') 
mergedData.head(3)

In [ ]:
mergedData.shape

In [ ]:
# Dropping unnecessary columns
mergedData = mergedData.drop(columns = ['homepage', 'title_x', 'title_y', 'status', 'production_countries'])
mergedData.head(3)

In [ ]:
mergedData.shape

In [ ]:
mergedData['overview'].head()

## NLP Feature Enginnering

Until now we have gathered the datasets, merged credits & movies dataset with 'id' as key area...

From the next code, we are going to predict the movies for recommendation...

TF-IDF : Term Frequency - Inverse Document Frquency
from scikit-learn it is imported. 

It is used to convert the text to numbers, where the ML can understands it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(stop_words = 'english', ngram_range = (1, 3), min_df = 3, analyzer = 'word')

# stop_words: the more recurrent words like 'a', 'the', 'is' are removed to improve the signal quality
# ngram_range: 1-word = 1-ngram, 2-words= 2-ngram .....
# min_df: is the minimum number of times the word  present in dataset
# analyzer: tokenization is done at word level rather than charater level

In [ ]:
# Filling Nan with empty string in overview column
mergedData['overview'] = mergedData['overview'].fillna(' ')

# Constructing tfidf_matrix by fitting and transform the data
tfidf_matrix = tfidf.fit_transform(mergedData['overview'])

tfidf_matrix.shape

We used linear_kernel over cosine_simalrity for comparison of word-vector embeddings, because since we are using TF-IDF, it is is already normalised...

So, using linear_kernel is same result and faster computation...

In [ ]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [ ]:
print(cosine_sim.shape)

In [ ]:
print(cosine_sim[1])

In [ ]:
# Constructing a reverse map of indices and movie title
indices = pd.Series(mergedData.index, index = mergedData['original_title']).drop_duplicates()

# pattern we obtain for this will be
# 'Batman' -> 0
# 'Joker' -> 1

Let's write a function to get the recommendation movies for a given movie

In [ ]:
def get_recommendations(title, cosine_sim = cosine_sim):
    # getting the index of the movie matching the title
    idx = indices[title]

    # get pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    #sort the movies based on the similarity scores and higher scores in top
    sim_scores = sorted(sim_scores, key = lambda x : x[1], reverse = True)

    #get the scores of the 10 most similar movies
    sim_scores = sim_scores[1 : 11]

    #get the movies indices
    movie_indices = [i[0] for i in sim_scores]

    #return top 10 most similar movies
    return mergedData['original_title'].iloc[movie_indices]

In [ ]:
# Lets get recommendation for Avatar movie
get_recommendations('Avatar')

In [ ]:
get_recommendations('The Dark Knight Rises')